In [ ]:
# Install required dependencies
!pip install -q kaggle-benchmarks numpy

# Error Detection Benchmark

Tests ability to detect and localize errors in reasoning chains.

**Cognitive Science**: Yeung & Summerfield (2012)
**Human d'**: 1.5–3.0
**Score**: F1 + localization + confidence calibration

In [ ]:
"""
Error Detection benchmark reasoning chains dataset.

Contains math/logic problems with step-by-step solutions.
Some solutions are correct; others have deliberate errors injected
at specific steps. The model must identify:
1. Whether an error exists (binary)
2. Which step contains the error
3. Confidence in its judgment

Categories:
- MATH: Arithmetic and algebra problems
- LOGIC: Logical deduction problems  
- PROBABILITY: Probability/combinatorics
"""

REASONING_CHAINS = [
    # === CORRECT CHAINS ===
    {
        "id": "C01",
        "problem": "Solve for x: 3x + 7 = 22",
        "steps": [
            "Step 1: Subtract 7 from both sides: 3x = 22 - 7 = 15",
            "Step 2: Divide both sides by 3: x = 15 / 3 = 5",
            "Step 3: Check: 3(5) + 7 = 15 + 7 = 22 ✓",
        ],
        "final_answer": "x = 5",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "C02",
        "problem": "What is the probability of rolling two dice and getting a sum of 7?",
        "steps": [
            "Step 1: Total possible outcomes when rolling two dice = 6 × 6 = 36",
            "Step 2: Favorable outcomes for sum of 7: (1,6), (2,5), (3,4), (4,3), (5,2), (6,1) = 6 outcomes",
            "Step 3: Probability = favorable / total = 6/36 = 1/6",
        ],
        "final_answer": "1/6",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "C03",
        "problem": "If all roses are flowers, and some flowers are red, can we conclude that some roses are red?",
        "steps": [
            "Step 1: Premise 1: All roses are flowers (roses ⊆ flowers)",
            "Step 2: Premise 2: Some flowers are red (flowers ∩ red ≠ ∅)",
            "Step 3: The red flowers could be non-rose flowers (e.g., tulips, poppies)",
            "Step 4: We cannot conclude that any roses are red — the conclusion does not follow",
        ],
        "final_answer": "No, we cannot conclude that some roses are red",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "C04",
        "problem": "Find the area of a triangle with base 12 cm and height 8 cm.",
        "steps": [
            "Step 1: Area formula for a triangle: A = (1/2) × base × height",
            "Step 2: A = (1/2) × 12 × 8",
            "Step 3: A = (1/2) × 96 = 48 cm²",
        ],
        "final_answer": "48 cm²",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "C05",
        "problem": "How many ways can 5 people be seated in a row?",
        "steps": [
            "Step 1: The first seat can be filled by any of 5 people",
            "Step 2: The second seat by any of the remaining 4",
            "Step 3: Continuing: 3, then 2, then 1",
            "Step 4: Total = 5! = 5 × 4 × 3 × 2 × 1 = 120",
        ],
        "final_answer": "120",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },

    # === CHAINS WITH ERRORS ===
    {
        "id": "E01",
        "problem": "Solve for x: 2(x + 3) = 16",
        "steps": [
            "Step 1: Distribute the 2: 2x + 3 = 16",  # ERROR: should be 2x + 6
            "Step 2: Subtract 3 from both sides: 2x = 13",
            "Step 3: Divide by 2: x = 6.5",
            "Step 4: Check: 2(6.5 + 3) = 2(9.5) = 19 ≠ 16, so let me recheck... Actually 2(6.5 + 3) = 19. Hmm, that's close to 16.",
        ],
        "final_answer": "x = 6.5",
        "has_error": True,
        "error_step": 1,
        "error_description": "Distribution error: 2(x+3) should give 2x + 6, not 2x + 3",
        "difficulty": 1,
    },
    {
        "id": "E02",
        "problem": "What is the probability of getting at least one head in 3 coin flips?",
        "steps": [
            "Step 1: P(at least one head) = 1 - P(no heads) = 1 - P(all tails)",
            "Step 2: P(all tails) = (1/2)³ = 1/6",  # ERROR: should be 1/8
            "Step 3: P(at least one head) = 1 - 1/6 = 5/6",
        ],
        "final_answer": "5/6",
        "has_error": True,
        "error_step": 2,
        "error_description": "Calculation error: (1/2)³ = 1/8, not 1/6",
        "difficulty": 1,
    },
    {
        "id": "E03",
        "problem": "If it rains, the ground is wet. The ground is wet. Did it rain?",
        "steps": [
            "Step 1: Premise: If rain → wet ground",
            "Step 2: Observation: The ground is wet",
            "Step 3: Since wet ground always comes from rain, it must have rained",  # ERROR: affirming the consequent
            "Step 4: Therefore, it rained",
        ],
        "final_answer": "Yes, it rained",
        "has_error": True,
        "error_step": 3,
        "error_description": "Affirming the consequent fallacy: the ground could be wet for other reasons (sprinkler, flood, etc.)",
        "difficulty": 2,
    },
    {
        "id": "E04",
        "problem": "Simplify: (x² - 9) / (x - 3)",
        "steps": [
            "Step 1: Factor the numerator: x² - 9 = (x - 3)(x - 3)",  # ERROR: should be (x-3)(x+3)
            "Step 2: Cancel (x - 3): (x - 3)(x - 3) / (x - 3) = x - 3",
            "Step 3: Result: x - 3 (for x ≠ 3)",
        ],
        "final_answer": "x - 3",
        "has_error": True,
        "error_step": 1,
        "error_description": "Factoring error: x² - 9 = (x-3)(x+3), not (x-3)(x-3). The correct simplification is x + 3.",
        "difficulty": 1,
    },
    {
        "id": "E05",
        "problem": "A train travels 120 km in 1.5 hours. Then it travels 80 km in 1 hour. What is the average speed for the entire trip?",
        "steps": [
            "Step 1: Speed for leg 1: 120/1.5 = 80 km/h",
            "Step 2: Speed for leg 2: 80/1 = 80 km/h",
            "Step 3: Average speed = (80 + 80) / 2 = 80 km/h",  # ERROR: should use total distance / total time
        ],
        "final_answer": "80 km/h",
        "has_error": True,
        "error_step": 3,
        "error_description": "Average speed should be total distance / total time = 200/2.5 = 80 km/h. In this case the answer happens to be correct by coincidence, but the method is wrong (averaging speeds is incorrect in general).",
        "difficulty": 2,
    },
    {
        "id": "E06",
        "problem": "How many diagonals does a hexagon have?",
        "steps": [
            "Step 1: Formula for diagonals of an n-gon: n(n-3)/2",
            "Step 2: For hexagon, n = 6: 6(6-3)/2 = 6 × 3/2 = 18/2 = 9",
        ],
        "final_answer": "9",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "E07",
        "problem": "Convert 5/8 to a percentage.",
        "steps": [
            "Step 1: To convert a fraction to a percentage, multiply by 100",
            "Step 2: 5/8 × 100 = 500/8 = 62.5%",
        ],
        "final_answer": "62.5%",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "E08",
        "problem": "What is the derivative of f(x) = x³ + 2x² - 5x + 1?",
        "steps": [
            "Step 1: Apply power rule term by term",
            "Step 2: d/dx(x³) = 3x²",
            "Step 3: d/dx(2x²) = 4x",
            "Step 4: d/dx(-5x) = -5",
            "Step 5: d/dx(1) = 1",  # ERROR: derivative of constant is 0
            "Step 6: f'(x) = 3x² + 4x - 5 + 1 = 3x² + 4x - 4",
        ],
        "final_answer": "f'(x) = 3x² + 4x - 4",
        "has_error": True,
        "error_step": 5,
        "error_description": "The derivative of a constant (1) is 0, not 1. Correct answer: f'(x) = 3x² + 4x - 5",
        "difficulty": 2,
    },
    {
        "id": "E09",
        "problem": "A bag contains 3 red and 5 blue balls. Two balls are drawn without replacement. What is the probability both are red?",
        "steps": [
            "Step 1: P(first red) = 3/8",
            "Step 2: After drawing one red, remaining: 2 red, 5 blue = 7 total",
            "Step 3: P(second red | first red) = 2/7",
            "Step 4: P(both red) = (3/8) × (2/7) = 6/56 = 3/28",
        ],
        "final_answer": "3/28",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "E10",
        "problem": "Evaluate: log₂(8) + log₂(4)",
        "steps": [
            "Step 1: log₂(8) = 3 (since 2³ = 8)",
            "Step 2: log₂(4) = 2 (since 2² = 4)",
            "Step 3: By the log addition rule: log₂(8) + log₂(4) = log₂(8 × 4) = log₂(32) = 5",
        ],
        "final_answer": "5",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    {
        "id": "E11",
        "problem": "Find the sum of the interior angles of a pentagon.",
        "steps": [
            "Step 1: Formula: sum of interior angles = (n-2) × 180°",
            "Step 2: For pentagon, n = 5: (5-2) × 180° = 3 × 180° = 480°",  # ERROR: 3 × 180 = 540
        ],
        "final_answer": "480°",
        "has_error": True,
        "error_step": 2,
        "error_description": "Arithmetic error: 3 × 180 = 540, not 480",
        "difficulty": 1,
    },
    {
        "id": "E12",
        "problem": "All cats are mammals. All mammals are warm-blooded. Therefore?",
        "steps": [
            "Step 1: Cats ⊆ Mammals (all cats are mammals)",
            "Step 2: Mammals ⊆ Warm-blooded (all mammals are warm-blooded)",
            "Step 3: By transitivity: Cats ⊆ Warm-blooded",
            "Step 4: Therefore, all cats are warm-blooded",
        ],
        "final_answer": "All cats are warm-blooded",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 1,
    },
    # Additional error chains for balance (targeting ~50/50 ratio)
    {
        "id": "E13",
        "problem": "What is the probability of drawing two aces in a row from a standard deck (without replacement)?",
        "steps": [
            "Step 1: P(first ace) = 4/52 = 1/13",
            "Step 2: After drawing one ace, 3 aces remain in 51 cards",
            "Step 3: P(second ace | first ace) = 3/52",  # ERROR: should be 3/51
            "Step 4: P(both aces) = (4/52) × (3/52) = 12/2704 = 3/676",
        ],
        "final_answer": "3/676",
        "has_error": True,
        "error_step": 3,
        "error_description": "Should be 3/51 (not 3/52) since one card has been removed. Correct answer: 12/2652 = 1/221",
        "difficulty": 2,
    },
    {
        "id": "E14",
        "problem": "Solve: |2x - 6| = 10",
        "steps": [
            "Step 1: |2x - 6| = 10 means either 2x - 6 = 10 or 2x - 6 = -10",
            "Step 2: Case 1: 2x - 6 = 10 → 2x = 16 → x = 8",
            "Step 3: Case 2: 2x - 6 = -10 → 2x = -4 → x = -2",
            "Step 4: Solutions: x = 8 or x = -2",
        ],
        "final_answer": "x = 8 or x = -2",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
    {
        "id": "E15",
        "problem": "What is the volume of a sphere with radius 3 cm?",
        "steps": [
            "Step 1: Volume formula: V = (4/3)πr²",  # ERROR: should be r³
            "Step 2: V = (4/3) × π × 3² = (4/3) × π × 9 = 12π",
            "Step 3: V ≈ 12 × 3.14159 ≈ 37.7 cm³",
        ],
        "final_answer": "37.7 cm³",
        "has_error": True,
        "error_step": 1,
        "error_description": "Volume formula should be (4/3)πr³, not (4/3)πr². Correct: (4/3)π(27) = 36π ≈ 113.1 cm³",
        "difficulty": 1,
    },
    {
        "id": "E16",
        "problem": "If the sequence follows the pattern: 2, 6, 18, 54, ... what is the 6th term?",
        "steps": [
            "Step 1: Find the common ratio: 6/2 = 3",
            "Step 2: This is a geometric sequence with a₁ = 2, r = 3",
            "Step 3: General term: aₙ = a₁ × r^(n-1) = 2 × 3^(n-1)",
            "Step 4: a₆ = 2 × 3^5 = 2 × 243 = 486",
        ],
        "final_answer": "486",
        "has_error": False,
        "error_step": None,
        "error_description": None,
        "difficulty": 2,
    },
]


In [ ]:
"""
MetaCog Benchmark 4: Error Detection (Metacognitive Monitoring of Reasoning)

Tests the model's ability to detect errors in step-by-step reasoning chains.
This measures metacognitive monitoring during/after processing — a critical
component of self-correction capability.

Protocol:
1. Present a problem with a worked step-by-step solution
2. Ask model to review the solution and:
   a. Determine if there's an error (binary)
   b. If yes, identify which step contains the error
   c. Rate confidence in the judgment (0-100)
3. Score based on detection accuracy, localization, and confidence calibration

Cognitive Science Basis:
- Yeung & Summerfield (2012): Error monitoring and metacognition
- Nelson & Narens (1990): Monitoring of ongoing cognitive processes
- Related to "debugging" in education research

Metrics:
- Error detection F1 (binary: error present or not)
- Error localization accuracy (correct step identified)
- Confidence calibration (ECE of error detection confidence)
- Signal detection: d' and meta-d' for error detection

Shortcut Resistance:
- Mix of correct and incorrect chains prevents bias
- Errors vary in subtlety (arithmetic, logic, conceptual)
- Some "errors" are actually correct (tests false alarm rate)
- Confidence calibration penalizes overconfident wrong judgments
"""

import kaggle_benchmarks as kbench
from dataclasses import dataclass
import numpy as np
import re
import json
# REASONING_CHAINS defined above


@dataclass
class ErrorReview:
    """Model's review of a reasoning chain."""
    has_error: bool       # Does this chain contain an error?
    error_step: int       # Which step (1-indexed), or 0 if no error
    explanation: str      # Explanation of the error (or why it's correct)
    confidence: int       # 0-100 confidence in the judgment


# ─── Helpers ─────────────────────────────────────────────────────

def goodman_kruskal_gamma(x: list, y: list) -> float:
    n = len(x)
    concordant = 0
    discordant = 0
    for i in range(n):
        for j in range(i + 1, n):
            x_diff = x[i] - x[j]
            y_diff = y[i] - y[j]
            product = x_diff * y_diff
            if product > 0:
                concordant += 1
            elif product < 0:
                discordant += 1
    denom = concordant + discordant
    return (concordant - discordant) / denom if denom > 0 else 0.0


def compute_ece(confidences: list, accuracies: list, n_bins: int = 5) -> float:
    conf = np.array(confidences) / 100.0
    acc = np.array(accuracies, dtype=float)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    total = len(conf)
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        if i == 0:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf > lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / total) * abs(acc[mask].mean() - conf[mask].mean())
    return round(float(ece), 4)


def compute_f1(tp: int, fp: int, fn: int) -> float:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def compute_dprime(hit_rate: float, false_alarm_rate: float) -> float:
    """Compute d' (signal detection sensitivity)."""
    from scipy.stats import norm  # type: ignore
    # Clamp to avoid infinite values
    hr = min(max(hit_rate, 0.01), 0.99)
    far = min(max(false_alarm_rate, 0.01), 0.99)
    return float(norm.ppf(hr) - norm.ppf(far))


# ─── The Benchmark Task ────────────────────────────────────────────

@kbench.task(name="metacog_error_detection")
def metacog_error_detection(llm) -> float:
    """
    Error Detection Benchmark.

    Model reviews step-by-step reasoning chains and must:
    1. Detect whether an error exists
    2. Localize the error (which step)
    3. Rate confidence

    Score = 0.35 * detection_F1 + 0.25 * localization_accuracy
            + 0.20 * (1 - ECE) + 0.20 * confidence_gamma_norm

    Cognitive Science Basis: Yeung & Summerfield (2012), error monitoring.
    """
    results = []

    for chain in REASONING_CHAINS:
        # Format the chain for review
        steps_text = "\n".join(chain["steps"])
        prompt = (
            f"Review the following step-by-step solution for errors.\n\n"
            f"**Problem:** {chain['problem']}\n\n"
            f"**Solution:**\n{steps_text}\n\n"
            f"**Final answer:** {chain['final_answer']}\n\n"
            f"Carefully check each step. Does this solution contain an error?\n\n"
            f"Respond with ONLY a JSON object:\n"
            f'{{"has_error": true/false, "error_step": <step number or 0>, '
            f'"explanation": "<explain the error or why correct>", '
            f'"confidence": <0-100>}}'
        )

        with kbench.chats.new(f"review_{chain['id']}"):
            try:
                review = llm.prompt(prompt, schema=ErrorReview)
                pred_has_error = review.has_error
                pred_step = review.error_step
                confidence = max(0, min(100, review.confidence))
                explanation = review.explanation
            except Exception:
                raw = llm.prompt(prompt)
                try:
                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
                    pred_has_error = bool(parsed.get("has_error", False))
                    pred_step = int(parsed.get("error_step", 0))
                    confidence = max(0, min(100, int(parsed.get("confidence", 50))))
                    explanation = str(parsed.get("explanation", ""))
                except Exception:
                    # Crude fallback: look for keywords
                    raw_lower = raw.lower()
                    pred_has_error = any(w in raw_lower for w in ["error", "mistake", "incorrect", "wrong"])
                    pred_step = 0
                    confidence = 50
                    explanation = raw[:200]

        # Score this chain
        actual_has_error = chain["has_error"]
        actual_step = chain["error_step"]

        # Detection correctness
        detection_correct = pred_has_error == actual_has_error

        # Localization correctness (only relevant when error exists and was detected)
        localization_correct = False
        if actual_has_error and pred_has_error and actual_step is not None:
            localization_correct = pred_step == actual_step

        results.append({
            "id": chain["id"],
            "problem": chain["problem"][:60],
            "actual_has_error": actual_has_error,
            "pred_has_error": pred_has_error,
            "actual_step": actual_step,
            "pred_step": pred_step,
            "detection_correct": detection_correct,
            "localization_correct": localization_correct,
            "confidence": confidence,
            "explanation": explanation[:100],
            "difficulty": chain["difficulty"],
        })

    # ── Compute Metrics ──
    # Detection F1
    tp = sum(1 for r in results if r["actual_has_error"] and r["pred_has_error"])
    fp = sum(1 for r in results if not r["actual_has_error"] and r["pred_has_error"])
    fn = sum(1 for r in results if r["actual_has_error"] and not r["pred_has_error"])
    tn = sum(1 for r in results if not r["actual_has_error"] and not r["pred_has_error"])
    f1 = compute_f1(tp, fp, fn)

    # Localization accuracy (among correctly detected errors)
    error_chains = [r for r in results if r["actual_has_error"] and r["pred_has_error"]]
    if error_chains:
        localization_acc = sum(1 for r in error_chains if r["localization_correct"]) / len(error_chains)
    else:
        localization_acc = 0.0

    # Confidence calibration
    confidences = [r["confidence"] for r in results]
    detection_accuracies = [r["detection_correct"] for r in results]
    ece = compute_ece(confidences, detection_accuracies)

    # Confidence-accuracy gamma
    gamma = goodman_kruskal_gamma(confidences, [int(a) for a in detection_accuracies])
    gamma_norm = (gamma + 1) / 2

    # Signal detection (d')
    n_signal = sum(1 for r in results if r["actual_has_error"])
    n_noise = sum(1 for r in results if not r["actual_has_error"])
    hit_rate = tp / n_signal if n_signal > 0 else 0
    false_alarm_rate = fp / n_noise if n_noise > 0 else 0

    try:
        dprime = compute_dprime(hit_rate, false_alarm_rate)
    except Exception:
        dprime = 0.0

    # Composite score
    score = round(
        0.35 * f1 + 0.25 * localization_acc + 0.20 * (1 - ece) + 0.20 * gamma_norm,
        4
    )

    # ── Logging ──
    print(f"\n{'='*60}")
    print(f"ERROR DETECTION BENCHMARK RESULTS")
    print(f"{'='*60}")
    print(f"Chains reviewed: {len(REASONING_CHAINS)}")
    print(f"  With errors: {n_signal}")
    print(f"  Without errors: {n_noise}")
    print(f"\n--- Detection Performance ---")
    print(f"True Positives:  {tp}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print(f"True Negatives:  {tn}")
    print(f"Detection F1:    {f1:.4f}")
    print(f"Hit rate:        {hit_rate:.2%}")
    print(f"False alarm:     {false_alarm_rate:.2%}")
    print(f"d' (sensitivity):{dprime:+.3f}")

    print(f"\n--- Localization ---")
    print(f"Correctly localized: {sum(1 for r in error_chains if r['localization_correct'])}/{len(error_chains)}")
    print(f"Localization acc:    {localization_acc:.2%}")

    print(f"\n--- Metacognitive Metrics ---")
    print(f"ECE:             {ece:.4f}")
    print(f"Gamma:           {gamma:+.4f}")
    print(f"Mean confidence: {np.mean(confidences):.1f}%")
    print(f"Composite score: {score:.4f}")

    print(f"\n--- Per-Chain Results ---")
    for r in results:
        det = "✓" if r["detection_correct"] else "✗"
        loc = ""
        if r["actual_has_error"] and r["pred_has_error"]:
            loc = " LOC:✓" if r["localization_correct"] else f" LOC:✗(pred={r['pred_step']},actual={r['actual_step']})"
        err_label = "ERR" if r["actual_has_error"] else "OK "
        print(f"  {det} [{r['confidence']:3d}%] [{err_label}] {r['problem'][:45]}...{loc}")

    return score


# ─── Run ────────────────────────────────────────────────────────────
metacog_error_detection.run(llm=kbench.llm)
